# peer_report

> The morning report: what the peers did while the user was away.

In [ ]:
#| default_exp peer_report

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| hide
import os, tempfile
from fastcore.test import test_eq

In work together mode, sessions work while the user is away. This module reads the log of messages and tells the user what happened.

The table, the glossary and the action items that cannot wait come from code. The simple examples come from one `summary` model call. If that call fails, the report is still saved, with one line that says the examples are missing.

The module loads no model code until it needs it, so `ramabana report` stays light.

In [ ]:
#| export
import argparse, re, subprocess, sys, time
from datetime import datetime
from pathlib import Path
from ramabana.peers import peers_dir, read_log, registered, TOGETHER_HOURS, HOP_MAX, PEER_BUDGET

## Threads

`threads` joins the log rows of one conversation. A thread starts with an `ask`. Its state tells the user how it ended:

- `replied`: a reply with proof.
- `not proven`: a reply with no `proof`.
- `refused`: a reply that starts with `refused:`.
- `waiting`: no reply yet.
- `stopped`: a limit ended the thread.

In [ ]:
#| export
REFUSAL = re.compile(r'\s*refused:', re.I)
NO_ASK = 'the ask is from before this report'

def _name(f): return f.get('name') or f.get('key') or '?'

def _uniq(rows, k): return list(dict.fromkeys(i for r in rows for i in r.get(k, [])))

def _thread(tid, rows, stop, who):
    "The summary of thread `tid`: its envelope `rows`, the `refused` event `stop` if any, and `who`, the sessions by key."
    ask = next((r for r in rows if r['kind'] == 'ask' and r['id'] == tid), None)
    replies = [r for r in rows if r['kind'] == 'reply']
    last = replies[-1] if replies else {}
    state = ('stopped' if stop else 'waiting' if not replies else 'refused' if REFUSAL.match(last['text'])
             else 'replied' if _uniq(replies, 'proof') else 'not proven')
    peer = lambda k: who.get(k) or {'key': k}
    receiver = peer(ask['to']) if ask else {}
    return dict(id=tid, state=state, why=(stop or {}).get('why', ''),
                ask=ask['text'] if ask else NO_ASK if replies else stop['why'],
                asker=_name(ask['from']) if ask else _name(peer(replies[0]['to'])) if replies else '?',
                worker=_name(last['from']) if last else _name(receiver) if ask else '',
                folders=_uniq([r['from'] for r in replies], 'folders') or receiver.get('folders', []),
                changes=_uniq(replies, 'changes'), proof=_uniq(replies, 'proof'))

def threads(rows, known=None):
    "One dict per thread in log `rows`: `id, asker, worker, ask, state, why, folders, changes, proof`; `known` maps keys to sessions that `rows` do not show."
    who = {**(known or {}), **{r['from']['key']: r['from'] for r in rows if r.get('from', {}).get('key')}}
    envs = [r for r in rows if r.get('kind')]
    stops = {r['thread']: r for r in rows if r.get('event') == 'refused'}
    by = {}
    for r in envs: by.setdefault(r['thread'], []).append(r)
    return [_thread(t, by.get(t, []), stops.get(t), who) for t in dict.fromkeys([*by, *stops])
            if t in stops or any(r['kind'] in ('ask', 'reply') for r in by[t])]

A small log with an ask that got a reply with proof, one that got none, and an ask with no reply. The last thread was stopped by the hop limit.

In [ ]:
#| hide
_f = lambda k, n: {'key': k, 'name': n}
_e = lambda i, kind, frm, to, text, thread=None, **kw: dict(id=i, kind=kind, thread=thread or i, text=text, to=to, at=1., **{'from': frm}, **kw)
LOG = [_e('a', 'ask', _f('c1', 'Leela'), 'r2', 'Add tell'),
       _e('a2', 'reply', _f('r2', 'Ramabana'), 'c1', 'Done', 'a', changes=['cli.py'], proof=['12 passed']),
       _e('b', 'ask', _f('c1', 'Leela'), 'r2', 'Fix card'),
       _e('b2', 'reply', _f('r2', 'Ramabana'), 'c1', 'Done', 'b'),
       _e('c', 'ask', _f('c1', 'Leela'), 'r2', 'Wait'),
       _e('d', 'ask', _f('c1', 'Leela'), 'r2', 'Loop'),
       {'event': 'refused', 'key': 'r2', 'why': 'hop limit: 6 of 6 hops used', 'thread': 'd', 'at': 2.}]

In [ ]:
test_eq([t['state'] for t in threads(LOG)], ['replied', 'not proven', 'waiting', 'stopped'])

The state `not proven` is the one to watch: the worker says it is done and shows no proof.

In [ ]:
test_eq(threads(LOG)[1]['proof'], [])
test_eq(threads(LOG)[0]['worker'], 'Ramabana')

Only a reply that starts with `refused:` is a refusal. An ask from before the report, or a stop with no ask, gets no made-up asker.

In [ ]:
test_eq(threads([LOG[0], dict(LOG[1], text='refused: not my repo')])[0]['state'], 'refused')
test_eq(threads([LOG[0], dict(LOG[1], text="I can't reproduce it, but I fixed X")])[0]['state'], 'replied')
t, = threads([LOG[1]])
test_eq((t['asker'], t['worker'], t['ask']), ('c1', 'Ramabana', NO_ASK))
t, = threads([LOG[-1]])
test_eq((t['asker'], t['worker'], t['ask']), ('?', '', 'hop limit: 6 of 6 hops used'))

## Checked changes

A worker lists the `changes` it made. `checked_changes` looks in `git log` of the worker's folders since the mode started. An item that git does not show gets `(not in git)`. Without folders, nothing can be checked, so the items stay as they are.

In [ ]:
#| export
def _git(folder, *a):
    "The output of `git` run in `folder`; None when git is missing or `folder` is not a repo."
    try: r = subprocess.run(['git', '-C', str(folder), *a], capture_output=True, text=True)
    except OSError: return None
    return r.stdout if r.returncode == 0 else None

def checked_changes(t, since):
    "The `changes` of thread `t`, each marked `(not in git)` when no commit or file of the worker's repos since `since` matches it; no repo, no marks."
    since = f'--since={int(since)}'
    logs = [(_git(f, 'log', '--all', since, '--format=%H'), _git(f, 'log', '--all', since, '--name-only', '--format=')) for f in t['folders']]
    logs = [l for l in logs if None not in l]
    shas = '\n'.join(h for h, _ in logs).split()
    files = [l for _, n in logs for l in n.split('\n') if l]
    seen = lambda c: any(s.startswith(c) for s in shas) if re.fullmatch(r'[0-9a-f]{7,40}', c) else any(f == c or f.endswith('/' + c) or c.endswith('/' + f) for f in files)
    return [c if not logs or seen(c) else f'{c} (not in git)' for c in t['changes']]

A temporary repo with one commit. The real commit and file pass; the others are marked.

In [ ]:
with tempfile.TemporaryDirectory() as d:
    run = lambda *a: subprocess.run(['git', '-C', d, '-c', 'user.name=t', '-c', 'user.email=t@t', *a], capture_output=True, text=True).stdout.strip()
    run('init', '-q'); (Path(d)/'a.py').write_text('x'); run('add', '.'); run('commit', '-q', '-m', 'one')
    sha = run('rev-parse', 'HEAD')[:8]
    t = dict(folders=[d], changes=[sha, 'deadbeef1', 'a.py', 'gone.py'])
    test_eq(checked_changes(t, time.time() - 3600), [sha, 'deadbeef1 (not in git)', 'a.py', 'gone.py (not in git)'])
    test_eq(checked_changes(dict(folders=[], changes=['x.py']), 0), ['x.py'])
with tempfile.TemporaryDirectory() as d: test_eq(checked_changes(dict(folders=[d], changes=['x.py']), 0), ['x.py'])

## The table

`table` makes one Markdown row per thread. A `|` in the text is escaped, and a long ask is cut short.

In [ ]:
#| export
COLS = ('Asked by', 'Done by', 'What', 'Changed', 'Proof', 'State')

def _cell(s, n=80):
    "`s` made safe for one table cell."
    s = ' '.join(str(s).split()).replace('|', '\\|')
    return s if len(s) <= n else s[:n - 1] + '…'

def table(ts):
    "A Markdown table of the threads `ts`, one row each."
    row = lambda *c: '| ' + ' | '.join(c) + ' |'
    return '\n'.join([row(*COLS), row(*['---']*len(COLS)),
                      *(row(*map(_cell, (t['asker'], t['worker'] or '-', t['ask'], '; '.join(t['changes']) or '-', '; '.join(t['proof']) or '-', t['state']))) for t in ts)])

In [ ]:
test_eq(table(threads(LOG)).splitlines()[0], '| Asked by | Done by | What | Changed | Proof | State |')
test_eq(len(table(threads(LOG)).splitlines()), 6)
assert '| Leela | Ramabana | Fix card |' in table(threads(LOG))

## The glossary

The report uses a few terms that a new reader may not know. `glossary` gives the meaning of only the terms in a text.

In [ ]:
#| export
GLOSSARY = {
    'peer': 'Another session on this Mac: Claude Code, Codex or Ramabana.',
    'note': 'A message that needs no answer.',
    'ask': 'A message that asks a peer to do work.',
    'reply': 'The answer to an ask.',
    'thread': 'One ask and all the replies to it.',
    'hop': f'One message in a thread. A thread carries at most {HOP_MAX} messages: the ask and {HOP_MAX - 1} replies.',
    'work together mode': 'A time when peers may take asks from each other with no one at the keyboard.',
    'proof': 'A test or a check that shows the work is done.',
    'budget': f"The most that one session may do: {PEER_BUDGET['messages']} messages sent in {TOGETHER_HOURS} hours, and in work together mode {PEER_BUDGET['messages']} messages sent and {PEER_BUDGET['turns']} asks taken.",
}

def glossary(text):
    "The lines of `GLOSSARY` for the terms that `text` uses."
    return '\n'.join(f'- **{k}**: {v}' for k, v in GLOSSARY.items() if re.search(rf'\b{re.escape(k)}s?\b', text, re.I))

In [ ]:
g = glossary('One ask, one reply, one thread.')
test_eq(g.count('\n'), 2)
assert 'budget' not in g and '**thread**' in g

## Actions and examples

A waiting ask, a `not proven` thread, a refusal and a stopped thread always need the user. `derived_actions` writes each one as "what to do: where". The model adds more items and one plain sentence for each change.

In [ ]:
#| export
TODO = {'waiting': 'Get an answer to', 'not proven': 'Ask for proof of', 'refused': 'Look at the refusal of', 'stopped': 'Restart'}

def derived_actions(ts):
    "The action lines that the table alone shows: each one is `what to do: where`."
    where = lambda t: (t['worker'] or t['asker']) + (f" ({Path(t['folders'][0]).name})" if t['folders'] else '')
    return [f"- {TODO[t['state']]} '{_cell(t['ask'], 40)}'" + (f" ({t['why']})" if t['why'] else '') + f": {where(t)}" for t in ts if t['state'] in TODO]

SUMMARY_SP = """You write the morning report for a user who was away. Use simple technical English (ASD-STE100), grade 9 reading level.
You get a list of threads. Answer in two parts. First a line `EXAMPLES`, then one bullet for each change, one short sentence each.
Then a line `ACTIONS`, then one bullet for each extra thing the user must do, written as "what to do: where".
Always name a waiting ask, a thread with no proof and a stopped thread as an action. Write no other text."""

def _bullets(s): return [l if l.lstrip().startswith(('-', '*')) else f'- {l}' for l in map(str.strip, s.splitlines()) if l]

def summarise(ts, agent=None):
    "The simple examples and the action list for threads `ts`, from one `summary` model call."
    from ramabana.agent import mk_agent
    agent = agent or mk_agent()
    out = agent.oneshot('\n'.join(f"- {t['asker']} asked {t['worker'] or 'nobody'}: {t['ask']} (state: {t['state']}; changed: {'; '.join(t['changes']) or 'nothing'})" for t in ts), SUMMARY_SP, job='summary')
    ex, *ac = re.split(r'(?im)^\s*actions\s*:?\s*$', out or '', maxsplit=1) + ['']
    return '\n'.join(_bullets(re.sub(r'(?im)^\s*examples\s*:?\s*$', '', ex))), '\n'.join(_bullets(ac[0]))

The action lines of the small log. No model is called here.

In [ ]:
test_eq(derived_actions(threads(LOG)), ["- Ask for proof of 'Fix card': Ramabana", "- Get an answer to 'Wait': Ramabana", "- Restart 'Loop' (hop limit: 6 of 6 hops used): Ramabana"])

A stub agent shows how the model reply is split. Its `oneshot` is the only part of the agent that `summarise` uses.

In [ ]:
class _Agent:
    def oneshot(self, prompt, sp, job): return 'EXAMPLES\nRamabana added tell.\nACTIONS\nRead the diff: cli.py'
test_eq(summarise(threads(LOG), _Agent()), ('- Ramabana added tell.', '- Read the diff: cli.py'))

## The report

`report` joins the parts. The action list has the derived lines first, then the model's own lines without repeats. If the summary fails, the report says why in place of the examples. With no thread, the model is not called.

In [ ]:
#| export
MISSING = 'The examples are missing'

def report(since, cfg=None, agent=None, summary=None):
    "The Markdown report of the log since time `since`; `summary` is `summarise` unless given."
    d = peers_dir(cfg)
    known = {r['key']: r for r in registered(d)} | {r['from']['key']: r['from'] for r in read_log(d) if r.get('from', {}).get('key')}
    ts = [t | dict(changes=checked_changes(t, since)) for t in threads(read_log(d, since), known)]
    ex, ac, why = '', '', 'it gave no answer'
    try:
        if ts: ex, ac = (summary or summarise)(ts, agent)
        else: ex = 'No thread in this time.'
    except Exception as e: why = f'{type(e).__name__}: {e}'
    acts = '\n'.join(dict.fromkeys([*derived_actions(ts), *_bullets(ac)])) or 'Nothing to do.'
    body = f"{table(ts)}\n\n## What changed\n\n{ex or f'{MISSING}: the summary model failed: {why}'}\n\n## Your actions\n\n{acts}"
    return f"# Night of {datetime.fromtimestamp(since):%Y-%m-%d}\n\n{body}\n\n## Glossary\n\n{glossary(body)}\n"

A log in a temporary folder and a model that fails. The table, the actions and the glossary are still there, with the reason.

In [ ]:
from unittest.mock import patch
from ramabana.peers import log

In [ ]:
def _down(ts, agent=None): raise RuntimeError('down')
with tempfile.TemporaryDirectory() as d, patch.dict(os.environ, RAMABANA_PEERS=d):
    for r in LOG: log(d, r)
    out = report(0, summary=_down)
    assert out.startswith('# Night of ') and f'{MISSING}: the summary model failed: RuntimeError: down' in out
    assert "Restart 'Loop'" in out and '**proof**' in out

## Saving it

`save_report` writes the report to `reports/<date>-<HHMM>.md` in the peers folder, named by `since`, so two modes on one date do not share a file. `report_path` gives that name, so a hook can find the file. It builds the file again when the log has newer rows, or when the saved file lacks the examples.

In [ ]:
#| export
def report_path(since, cfg=None):
    "The file in the reports folder for the report since `since`."
    return peers_dir(cfg)/'reports'/f'{datetime.fromtimestamp(since):%Y-%m-%d-%H%M}.md'

def save_report(since, cfg=None, summary=None):
    "Write the report since `since` to the reports folder; a file with examples and newer than the whole log is kept."
    d, p = peers_dir(cfg), report_path(since, cfg)
    if p.exists() and MISSING not in p.read_text() and not read_log(d, p.stat().st_mtime): return p
    p.parent.mkdir(exist_ok=True)
    p.write_text(report(since, cfg, summary=summary))
    return p

A file with the examples is kept. A file that lacks them, a new log row, or another `since` makes a new file.

In [ ]:
ok = lambda ts, agent=None: ('- ok', '')
with tempfile.TemporaryDirectory() as d, patch.dict(os.environ, RAMABANA_PEERS=d):
    for r in LOG: log(d, r)
    p = save_report(0, summary=_down)
    assert MISSING in p.read_text() and p == report_path(0)
    test_eq(save_report(0, summary=ok), p)
    assert MISSING not in p.read_text()
    p.write_text('kept'); os.utime(p, (time.time(), time.time()))
    test_eq(save_report(0, summary=ok).read_text(), 'kept')
    log(d, dict(LOG[0], id='new', thread='new', at=time.time() + 5))
    assert save_report(0, summary=ok).read_text().startswith('# Night of')
    assert save_report(3600, summary=ok) != p

## The command

`ramabana report [--since T] [--save]` prints the report, or saves it and prints the file name. `T` is a number of seconds since 1970 or a date such as `2026-10-28 22:00`. Without `--since`, the report starts when the last work together mode started.

In [ ]:
#| export
def _since(s):
    "Time `s` as seconds since 1970: a number or an ISO date."
    try: return float(s)
    except ValueError: return datetime.fromisoformat(s).timestamp()

def report_cli(argv, cfg=None):
    "Run `ramabana report` with the arguments `argv`; the exit code."
    p = argparse.ArgumentParser(prog='ramabana report')
    p.add_argument('--since', help='start time: seconds since 1970 or YYYY-MM-DD[ HH:MM]')
    p.add_argument('--save', action='store_true', help='write the report to the reports folder')
    a = p.parse_args(argv)
    ends = [r['started'] for r in read_log(peers_dir(cfg)) if r.get('event') == 'together_end']
    try: since = _since(a.since) if a.since else ends[-1] if ends else time.time() - TOGETHER_HOURS*3600
    except ValueError: print(f'bad time {a.since!r}', file=sys.stderr); return 2
    print(save_report(since, cfg) if a.save else report(since, cfg))
    return 0

In [ ]:
with tempfile.TemporaryDirectory() as d, patch.dict(os.environ, RAMABANA_PEERS=d):
    test_eq(report_cli(['--since', 'soon']), 2)

`python -m ramabana.peer_report` runs the same command. The peer hooks run it with their own Python, so an older `ramabana` first on `PATH` does not matter.

In [ ]:
#| export
#| eval: false
if __name__ == '__main__': sys.exit(report_cli(sys.argv[1:]))